### UKB Bacterial Infection Cases

In [ ]:
# Imports here.
import numpy as np
import pandas as pd
import os
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings("ignore")

#Directory in Biowulf
os.chdir('/data/CARD/projects/UKB_NDD_virus/UKB_Files')

# Loading Files

In [ ]:
#Loading all the reference files for UKB
NDD_free = pd.read_csv('ALL_NDD_FREE_CONTROLS_AGE60PLUS.txt', delimiter='\t') #NDD free controls, subset of phenome, all European ancestry
phenome = pd.read_csv('covariates_phenome_to_use.txt', delimiter='\t') #All individuals
massive_ICD10 = pd.read_csv('massive_ICD10_ALL_table.txt', delimiter='\t', header = None) #ICD10 codes per individual

In [ ]:
massive_ICD10

In [ ]:
NDD_free

In [ ]:
phenome

In [ ]:
massive_ICD10[0].value_counts()

In [ ]:
#Loading all the disease files for UKB
AD = pd.read_csv("alzheimer_disease.txt", delimiter = '\t')
ALS = pd.read_csv('ALS.IDs', header = None) 
dementia = pd.read_csv('Dementia.IDs', header = None)
PD = pd.read_csv('parkinson_disease.txt', delimiter='\t')
vascular = pd.read_csv('Vascular.IDs', header = None)

In [ ]:
AD

# Adding Disease Column

In [ ]:
#Pick your NDD here
NDD_all = ["PD", "ALS", "vascular", "AD", "dementia"]
NDD_dfs = [PD, ALS, vascular, AD, dementia]

In [ ]:
#Add disease column to NDD free df
for ndd in NDD_all:
    NDD_free[ndd] = 0

#Drop FID, Batch, and European
NDD_free = NDD_free.drop(columns = ['FID', 'BATCH', "EUROPEAN"])

#Rename ID column
NDD_free = NDD_free.rename(columns = {'IID': 'ID'})
print("Number of controls:", len(NDD_free))
NDD_free.head()

In [ ]:
#Creating df of people with NDD
all_cases =[]
for ndd in NDD_all:
    i = NDD_all.index(ndd)
    NDD_df = NDD_dfs[i]
    NDD_df = NDD_df.rename(columns = {'eid': 0})
    NDD_list = list(NDD_df[0])
    has_NDD = phenome[phenome['IID'].isin(NDD_list)]
    has_NDD = has_NDD[has_NDD['EUROPEAN'] == 1] #Only select Europeans to match controls
    has_NDD = has_NDD.drop(columns = ['FID', 'BATCH', "EUROPEAN"]) #Drop FID, Batch, and European
    has_NDD = has_NDD.rename(columns = {'IID': 'ID'})#Rename ID column
    has_NDD[ndd] = 1
    print(f"Number of individuals with {ndd}: {len(has_NDD)}")
    all_cases.append(has_NDD)

has_NDD_all = pd.concat(all_cases,ignore_index=True)
has_NDD_all = has_NDD_all.fillna(0)
has_NDD_all.head()

In [ ]:
#Combine NDD_free and has_NDD
df = pd.concat([NDD_free, has_NDD_all])
df.head()

In [ ]:
df.AD.value_counts()

# Adding ICD10 Codes to dataframe

In [ ]:
search_terms = pd.read_csv('/data/CARD/projects/UKB_NDD_virus/FinnGen_Jan_2023/Codes_for_bacterial_sleep_intestinal - bacteria_codes.csv')
search_terms

In [ ]:
phenocode_list = list(search_terms['phenocode'].drop_duplicates())
phenocode_list

In [ ]:
ukb_code_list = list(search_terms['ICD'])
ukb_code_list

In [ ]:
unique_codes = []
for code in ukb_code_list:
    unique_codes.append(code.split(','))

clean = []
for i in unique_codes[0]:
    clean.append(i.strip())
print(clean)

In [ ]:
print("Unique codes:", len(unique_codes))
print("Phenocode_list:", len(phenocode_list))
phenocode_list

flat_list = []
for xs in unique_codes:
    for x in xs:
        flat_list.append(x.strip())

In [ ]:
# All flat_lists for each ICD10 Grouping used in final model 
flat_list = ['A480','A481', 'A483','A488','A490','A491','A492','A493','A498','A499']
flat_list1 = ['A040', 'A041', 'A043', 'A044', 'A045', 'A046', 'A047', 'A048', 'A049', 'A052', 'A054', 'A058', 'A059']
flat_list2 = ['A310', 'A311', 'A318', 'A319', 'A321', 'A327', 'A328', 'A329', 'A35', 'A363', 'A368', 'A369', 'A370', 'A379', 'A38', 'A390', 'A391', 'A392', 'A394', 'A398', 'A399', 'A400', 'A401', 'A402', 'A403', 'A408', 'A409', 'A410', 'A411', 'A412', 'A413', 'A414', 'A415', 'A418', 'A419', 'A421', 'A422', 'A427', 'A428', 'A429', 'A449', 'A46']
flat_list3 = ['B950', 'B951','B952', 'B953', 'B954', 'B955', 'B956', 'B957', 'B958','B960', 'B961', 'B962', 'B963', 'B964', 'B965', 'B966', 'B967', 'B968', 'B970', 'B971', 'B972', 'B973', 'B974', 'B975', 'B976', 'B977', 'B978']
flat_list4 = ['J150', 'J151', 'J152', 'J153', 'J154', 'J155', 'J156', 'J157', 'J158', 'J159', 'J170']
flat_list5 = ['G000', 'G001', 'G002', 'G003', 'G008', 'G009', 'G01']
flat_list6 = ['J09', 'J100', 'J101', 'J108', 'J110', 'J111', 'J118', 'J120', 'J121', 'J122', 'J123', 'J128', 'J129', 'J13', 'J14', 'J150', 'J151', 'J152', 'J153', 'J154', 'J155', 'J156', 'J157', 'J158', 'J159', 'J160', 'J168', 'J170', 'J171', 'J172', 'J173', 'J178', 'J180', 'J181', 'J182' 'J188', 'J189']
flat_list7 = ['A514', 'A521', 'A523', 'A527', 'A528', 'A530', 'A53.9']
flat_list8 = ['A150', 'A151', 'A152', 'A153','A154', 'A155', 'A156', 'A157', 'A159', 'A160', 'A161', 'A162', 'A163', 'A164', 'A165', 'A167', 'A169', 'A170', 'A178', 'A180', 'A181', 'A182', 'A183', 'A184', 'A185', 'A187', 'A188', 'A190', 'A191', 'A192', 'A198', 'A199']
flat_list9=['K020', 'K021', 'K022', 'K023', 'K025', 'K028', 'K029', 'K040', 'K041', 'K043', 'K044', 'K045', 'K046', 'K047', 'K048', 'K049', 'K050', 'K051', 'K052', 'K053', 'K054', 'K055', 'K056']

In [ ]:
#code simplification
group_names = ['unspec_bacterial_infections', 'intestinal_infections', 'other_bact','bacterial_agents_other','bacterial_pneum', 'bacterial_menin','all_pneum','syphilis','TB','oral_diseases']
group_lists =[flat_list, flat_list1,flat_list2,flat_list3, flat_list4, flat_list5, flat_list6, flat_list7, flat_list8, flat_list9]

for i in range(len(group_names)):
    name = group_names[i]
    icd_list = group_lists[i]
    sub = massive_ICD10[massive_ICD10[1].isin(icd_list)]
    sub = sub.drop_duplicates(subset=0, keep ="first")
    ids = sub[0].unique()
    df[name]=df["ID"].isin(ids).astype(int)

In [ ]:
df.columns

In [ ]:
#Checking that ICD10 columns were added
df = df[['ID', 'BIRTH_YEAR', 'TOWNSEND', 'AGE_OF_RECRUIT', 'GENETIC_SEX', 'AD','ALS','vascular', 'dementia','PD','unspec_bacterial_infections','other_bact','intestinal_infections', 'bacterial_agents_other','bacterial_pneum','bacterial_menin','all_pneum','syphilis','TB','oral_diseases']]
df

In [ ]:
df.oral_diseases.value_counts()

In [ ]:
unique_codes = flat_list
print("Unique codes:", len(unique_codes))
#print(unique_codes)
print("Phenocode_list:", len(phenocode_list))
#print(phenocode_list)
print(unique_codes[0])
print(phenocode_list[0])

In [ ]:
#Fill nan values with 0
df = df.fillna(0)
df.head()

In [ ]:
# Create lists for the regression
predictor_list = ['unspec_bacterial_infections', 'intestinal_infections', 'other_bact', 'bacterial_agents_other','bacterial_pneum','bacterial_menin','all_pneum','syphilis','TB', 'oral_diseases']
predictor_meaning = ['Unspecified bacterial infections', 'Intestinal bacterial infections', 'Other bacterial diseases', 'Other bacterial agents','Bacterial pneumonia', 'Bacterial meningitis', 'Pneumonia (All)', 'Syphilis', 'Tubercolosis','Dental diseases']

# Regressions

In [ ]:
# Now nothing left to do is run the regressions and call it a day. 
from statsmodels.stats.multitest import fdrcorrection
results = []

for ndd in NDD_all: #NDD_ all is the list from above with all ndds

    # for predictor in range(1, 10):
    for predictor in range(len(predictor_list)):
      predictor_name = predictor_list[predictor]
      predictor_description = predictor_meaning[predictor]
      this_formula = ndd + "~ df['" + predictor_list[predictor] + "']" + " + AGE_OF_RECRUIT + TOWNSEND + GENETIC_SEX"
      fitted = sm.formula.glm(formula=this_formula, family=sm.families.Binomial(), data=df).fit()
      beta_coef  = fitted.params.loc["df['" + predictor_name + "']"]
      beta_se  = fitted.bse.loc["df['" + predictor_name + "']"]
      p_val = fitted.pvalues.loc["df['" + predictor_name + "']"]
      odds_ratio = np.exp(fitted.params.loc["df['" + predictor_name + "']"])
      conf = fitted.conf_int().loc["df['" + predictor_name + "']"]
      m5, m95 = np.exp(conf)
      n = sum(df[predictor_name])
      df2 = df[df[predictor_name]==1]
      n_pairs = sum(df2[ndd])  
    
      print(predictor_name, odds_ratio, m5, m95, p_val, n_pairs, n)
      results.append((ndd, predictor_name, predictor_description, odds_ratio, m5, m95, p_val, n_pairs, n))

output = pd.DataFrame(results, columns=('NDD','CODE', 'DESCRIPTION','odds_ratio', 'ci_min', "ci_max", 'P_VAL', "N_pairs", "N"))

In [ ]:
#Only looking at codes that have at least 3 pairings
output = output[output['N_pairs'] > 2]
output

In [ ]:
#Adding FDR Correction

#Sort P-values
output = output.sort_values(by = "P_VAL")

#Drop Nan-values
output = output.dropna()

#FDR Correction
rejected, p_corr = fdrcorrection(output['P_VAL'], is_sorted=True)
output['P_CORR'] = p_corr
output['SIGNIFICANT'] = rejected

In [ ]:
##### Check results
output